# 在工具中访问 Static Runtime Context

工具通过注入参数 `ToolRuntime` 访问 Context：

```python
context = runtime.context
```

`ToolRuntime` 参数不会暴露在工具的 JSON Schema 中，因此模型既不需要填写，也不能自行选择其中的 `user_id`、数据库连接等可信依赖。

## 使用 Context 注入 Repository 对象

下面用普通 Python 对象模拟数据库 Repository。真实产品中可以替换为数据库连接、Session、HTTP Client 或业务服务对象。

`frozen=True` 只能阻止 Context 字段被重新赋值；它不会让 Repository 内部的数据自动变成不可变数据。这里的核心约定仍然是：一次 Agent 运行期间不要替换 Context 中的依赖。

In [ ]:
from dataclasses import dataclass


class UserRepository:
    """使用内存字典模拟用户数据库。"""

    def __init__(self) -> None:
        self._profiles = {
            "user-1": {"name": "小花", "department": "研发部"},
            "user-2": {"name": "小明", "department": "市场部"},
        }

    def get_profile(self, user_id: str) -> dict | None:
        return self._profiles.get(user_id)


@dataclass(frozen=True)
class AppContext:
    user_id: str
    user_role: str
    repository: UserRepository


repository = UserRepository()


## 定义访问 Context 的工具

工具不接收模型传入的 `user_id`，而是从可信的 Runtime Context 获取当前用户身份，再调用 Context 中的 Repository。

In [ ]:
from langchain.agents import AgentState
from langchain.tools import ToolRuntime, tool


@tool
def get_current_user_profile(
    runtime: ToolRuntime[AppContext, AgentState],
) -> str:
    """查询当前登录用户的个人资料。"""
    context = runtime.context
    profile = context.repository.get_profile(context.user_id)
    return str(profile) if profile else "没有找到当前用户资料"


@tool
def get_current_user_permissions(
    runtime: ToolRuntime[AppContext, AgentState],
) -> str:
    """查询当前登录用户拥有的操作权限。"""
    role = runtime.context.user_role
    if role == "admin":
        return "read, write, manage_users"
    return "read"


## 查看模型能够看到的工具参数

下面的 Schema 中不会出现 `runtime`、`user_id` 或 `repository`。这些参数由 LangChain 在执行工具时注入，不由模型生成。

In [ ]:
print(get_current_user_profile.args_schema.model_json_schema())
print(get_current_user_permissions.args_schema.model_json_schema())


## 创建并调用 Agent

同一个工具没有显式业务参数，但每次运行会根据 Context 中的当前用户访问不同数据。

In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
)

agent = create_agent(
    model=model,
    tools=[get_current_user_profile, get_current_user_permissions],
    context_schema=AppContext,
    system_prompt=(
        "你是企业内部助手。用户询问自己的资料或权限时，"
        "必须调用相应工具，不要要求用户提供 user_id。"
    ),
)


In [ ]:
contexts = [
    AppContext("user-1", "member", repository),
    AppContext("user-2", "admin", repository),
]

for context in contexts:
    result = agent.invoke(
        {"messages": "请查询我的个人资料和操作权限。"},
        context=context,
    )
    print(f"\n[{context.user_id}]")
    for message in result["messages"]:
        message.pretty_print()


## Context 参数与普通工具参数

| 数据来源 | 示例 | 谁提供 | 是否出现在工具 Schema |
| --- | --- | --- | --- |
| 普通工具参数 | 查询关键词、目标城市 | 模型根据对话生成 | 是 |
| Static Runtime Context | 当前用户 ID、角色、数据库连接 | 应用在调用开始时注入 | 否 |

安全相关的用户身份、租户和权限不应该设计成普通工具参数，否则模型可能填入其他用户 ID。Context 可以把“模型可以决定的参数”和“应用可信的依赖”清楚分开。

## 小结

- 工具通过 `runtime.context` 读取静态 Context。
- `ToolRuntime` 是注入参数，不暴露给模型。
- Context 适合用户身份、数据库连接和客户端对象；会变化的对话数据应放在 State，需要跨会话持久化的数据应放在 Store。
- Context 中的连接对象应由应用负责创建、复用和关闭，Agent 不负责管理其生命周期。